[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/r1-q2/02_hub_identification.ipynb)

# R1-Q2 Week 3 — Identify hub genes in stress-relevant modules

This notebook identifies hubs in each per-tissue network, restricted to the modules that are most relevant to stress response.

By the end of this notebook you will have:

- A defended list of stress-relevant modules for shoot and root, produced by your pre-committed criterion.
- Per-tissue hub gene tables ranked by kME (`shoot_hubs.parquet`, `root_hubs.parquet`).
- A cross-tissue overlap picture showing which hubs appear in both shoot and root — the strongest tissue-independent candidates.
- A `hub_summary.json` ready as Week 3 draft-presentation input.

In [ ]:
# There are three patterns for installing irilab2026 from GitHub, depending on your needs. 

# The first is for a fresh, complete runtime. This installs irilab2026 and every dependency it declares. 
# It skips anything pip already sees as installed. This is ideal for a new environment or when you want 
# to be sure everything is up to date.

!pip install git+https://github.com/geraldmc/irilab2026.git@main
 
# The second is for code iteration only. It reinstalls irilab2026 itself, ignoring its dep list entirely. 
# The dep tree is left exactly as it is. This is ideal for iterating on irilab2026 itself, when you know 
# the deps are already satisfied and don't want to waste time reinstalling them.
# 
# !pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main

# The third is for developers who want to work with the code directly and see changes reflected 
# immediately without reinstalling.

#!pip install --force-reinstall --no-deps git+https://github.com/geraldmc/irilab2026.git@main
#!pip install git+https://github.com/geraldmc/irilab2026.git@main

## 1) Load the networks and the sample metadata

Notebook 1 built per-tissue co-expression networks with PyWGCNA and saved them as `shoot.p` and `root.p`. This section loads both pickles, loads the AtGenExpress sample metadata separately, and confirms that the network shapes and the sample IDs line up.

The metadata is loaded separately because PyWGCNA's `saveWGCNA()` doesn't serialize the sample-level metadata that N1 attached. It's needed in Section 2, where the first analytical step correlates each module's eigengene against per-stress one-hot indicators built from the metadata.

In [ ]:
# 1.1) Imports, output directory, pre-commit load
import json
import pickle
import pandas as pd
import irilab2026 as iri
import PyWGCNA  # required for unpickling N1's WGCNA objects

iri.setup()

OUTPUT_DIR = iri.output_dir("r1_q2")
print(f"Output directory: {OUTPUT_DIR}")

precommit_path = OUTPUT_DIR / "precommit.json"
try:
    with open(precommit_path) as f:
        precommit = json.load(f)
except FileNotFoundError:
    raise FileNotFoundError(
        f"\nCould not find your pre-commit file.\n"
        f"  Expected at: {precommit_path}\n\n"
        f"Run `00_question_orientation.ipynb` to completion first — "
        f"its final section writes `precommit.json` to this location.\n"
    ) from None

print(f"Loaded pre-commit with {len(precommit)} top-level entries.")

In [ ]:
# Loads the WGCNA pickles
shoot_path = OUTPUT_DIR / "shoot.p"
root_path = OUTPUT_DIR / "root.p"

with open(shoot_path, "rb") as f:
    shoot_wgcna = pickle.load(f)
with open(root_path, "rb") as f:
    root_wgcna = pickle.load(f)

print(f"Loaded shoot.p  ({shoot_path.stat().st_size:>10,} bytes)")
print(f"Loaded root.p   ({root_path.stat().st_size:>10,} bytes)")

In [ ]:
# Load AtGenExpress sample metadata. PyWGCNA's saveWGCNA() doesn't persist
# the obs-level metadata attached in N1's S5.1, so we re-load it here from
# the same source N0b used.
metadata = iri.atgenexpress_metadata()

shoot_meta = metadata.loc[shoot_wgcna.datExpr.obs.index]
root_meta = metadata.loc[root_wgcna.datExpr.obs.index]

print(f"shoot metadata: {shoot_meta.shape}  columns={list(shoot_meta.columns)}")
print(f"root  metadata: {root_meta.shape}")

In [ ]:
# Confirm shapes, module counts, and metadata alignment.
# PyWGCNA convention: datExpr.obs = samples, datExpr.var = genes (with module columns).
for label, wgcna, meta in [("shoot", shoot_wgcna, shoot_meta),
                            ("root",  root_wgcna,  root_meta)]:
    n_samples, n_genes = wgcna.datExpr.shape
    n_modules = wgcna.datExpr.var["moduleColors"].nunique()
    n_stress_labels = meta["stress"].notna().sum()

    print(f"{label}:")
    print(f"  network:   {n_samples} samples × {n_genes} genes, {n_modules} modules")
    print(f"  metadata:  {meta.shape[0]} samples, {n_stress_labels} with stress labels")
    print()

Both networks loaded with the shape Notebook 1 reported (124 samples × ~5,700 genes, 9 modules per tissue) and the sample metadata is on the `obs` axis. The next section applies the module-relevance pre-commit to find which modules are stress-relevant per tissue.

## 2) Identify stress-relevant modules per tissue

Not every module in a co-expression network responds to stress. Many capture baseline biology — housekeeping pathways, growth processes, tissue-specific defaults — that varies across samples for reasons unrelated to the stress treatment. To find hubs that are actually about stress response, we first filter the modules.

A module is **stress-relevant for stress *S*** if the absolute Pearson correlation between its eigengene and the one-hot indicator for stress *S* is at least **0.3**. This filter is applied per tissue.

The threshold of 0.3 follows the WGCNA convention from Langfelder & Horvath's tutorial, where module–trait correlations of \|r\| ≥ 0.2–0.3 are the standard significance benchmark. It is a convention threshold, not a value tuned to this dataset.

Two consequences worth flagging upfront:

- Some stresses may yield no qualifying module in a given tissue. That's a result, not a defect — it tells you the tissue has no co-expression module strongly tracking that stress at the module-eigengene level.
- Some modules may qualify for multiple stresses. Per-stress candidate lists are not disjoint by design; modules tracking biologically coupled stresses (e.g., salt and osmotic) should show up in both lists.

The eight stresses are correlated against; `control` is the implicit baseline and is not included as its own column.

In [ ]:
# Build per-tissue one-hot indicators for the eight stresses (excluding control).
shoot_onehot = pd.get_dummies(shoot_meta["stress"], dtype=int).drop(columns=["control"])
root_onehot = pd.get_dummies(root_meta["stress"], dtype=int).drop(columns=["control"])

print(f"shoot one-hot: {shoot_onehot.shape}  (samples × stresses)")
print(f"root  one-hot: {root_onehot.shape}")
print(f"\nStresses: {list(shoot_onehot.columns)}")

In [ ]:
# For each tissue, correlate every module eigengene against every one-hot stress.
# MEs is (samples × modules); one-hot is (samples × stresses).
# Result is a (modules × stresses) Pearson correlation matrix.

def module_stress_correlations(wgcna_obj, onehot_df):
    MEs = wgcna_obj.MEs
    common = MEs.index.intersection(onehot_df.index)
    MEs = MEs.loc[common]
    onehot = onehot_df.loc[common]
    return pd.DataFrame(
        {stress: MEs.corrwith(onehot[stress]) for stress in onehot.columns}
    )

shoot_corr = module_stress_correlations(shoot_wgcna, shoot_onehot)
root_corr = module_stress_correlations(root_wgcna, root_onehot)

print(f"shoot correlation matrix: {shoot_corr.shape}  (modules × stresses)")
print(f"root  correlation matrix: {root_corr.shape}")

In [ ]:
# Threshold hardcoded per the May 16 closeout lock.
# N0's precommit.json will be updated in a separate pass; until then this
# value is not read from precommit["hub_identification"].
MODULE_RELEVANCE_THRESHOLD = 0.3


def candidate_modules(corr_df, threshold=MODULE_RELEVANCE_THRESHOLD):
    """Per-stress list of (module, r) pairs above the threshold, sorted by |r|."""
    candidates = {}
    for stress in corr_df.columns:
        s = corr_df[stress]
        qualifying = s[s.abs() >= threshold]
        order = qualifying.abs().sort_values(ascending=False).index
        candidates[stress] = [(m, float(r)) for m, r in qualifying.loc[order].items()]
    return candidates


shoot_candidates = candidate_modules(shoot_corr)
root_candidates = candidate_modules(root_corr)

In [ ]:
def print_candidates(tissue, candidates):
    print(f"=== {tissue} ===")
    for stress, mods in candidates.items():
        if not mods:
            print(f"  {stress:<10}  (no qualifying module)")
        else:
            parts = [f"{m.replace('ME', ''):>12} ({r:+.2f})" for m, r in mods]
            print(f"  {stress:<10}  {', '.join(parts)}")
    print()


print_candidates("shoot", shoot_candidates)
print_candidates("root", root_candidates)

The per-stress candidate lists above are the input to Section 3, which computes intramodular connectivity (kME) for every gene in every stress-relevant module. The same module can appear in multiple stress lists; that's expected — a module that tracks both salt and osmotic stress is meaningful biology, not a duplicate to deduplicate.

Section 3 doesn't deduplicate either. A gene that's a hub in a salt-relevant module *and* an osmotic-relevant module shows up in both per-stress hub lists in Section 4. The final per-tissue hub table assembled at the end of the notebook collapses the redundancy.

## 3) Score genes within stress-relevant modules

For each stress-relevant module identified in Section 2, we now compute every member gene's intramodular connectivity score — the basis for ranking genes by how central they are to the module's expression pattern.

**Pre-commit (locked on the question page):** The hub-ness metric is **kME** — the Pearson correlation between a gene's expression and its module's eigengene. A high-kME gene tracks the module's overall pattern closely; a low-kME gene drifts more.

Three reasons kME is locked rather than presented as a choice:

1. **WGCNA-native.** The package is designed around the module eigengene as the central abstraction; kME is the metric that measures how well a gene tracks that abstraction.
2. **Biologically interpretable.** A gene with kME = 0.9 in a module is one whose expression pattern is 90% correlated with the module's summary — directly readable as biology. Topological centrality measures (degree, betweenness, eigenvector) require translation.
3. **Computed natively by PyWGCNA's data structures.** Module eigengenes are already stored; kME is a one-line correlation away.

Kumar et al. (2023), cited on the question page, used kME to identify 117 central transcription factors in an *Arabidopsis* drought co-expression study — the rationale-level precedent for this question.

This section produces a per-tissue table: every gene in a stress-relevant module, with its module assignment and kME. Section 4 applies the hub threshold.

In [ ]:
# Section 2 produced per-stress candidate lists; collapse to a single set of
# stress-relevant modules per tissue. The ME prefix is stripped here so the
# module names match wgcna.datExpr.var["moduleColors"].

shoot_relevant_modules = {m[2:] for mods in shoot_candidates.values() for m, _ in mods}
root_relevant_modules  = {m[2:] for mods in root_candidates.values()  for m, _ in mods}

print(f"shoot stress-relevant modules ({len(shoot_relevant_modules)}): "
      f"{sorted(shoot_relevant_modules)}")
print(f"root  stress-relevant modules ({len(root_relevant_modules)}): "
      f"{sorted(root_relevant_modules)}")

In [ ]:
def compute_kme_table(wgcna_obj, relevant_modules):
    """
    For every gene assigned to any module in `relevant_modules`, compute kME:
    the Pearson correlation between the gene's expression across samples and
    its module's eigengene across samples.

    Returns
    -------
    pd.DataFrame indexed by probe_id with columns: module, kME.
    """
    expr = wgcna_obj.datExpr.to_df()              # samples × genes
    gene_module = wgcna_obj.datExpr.var["moduleColors"]
    eigengenes = wgcna_obj.MEs                    # samples × modules (ME-prefixed)

    rows = []
    for module in sorted(relevant_modules):
        genes_in_module = gene_module[gene_module == module].index
        if len(genes_in_module) == 0:
            continue

        me = eigengenes[f"ME{module}"]
        kme = expr[genes_in_module].corrwith(me)

        for probe_id, kme_val in kme.items():
            rows.append({
                "probe_id": probe_id,
                "module":  module,
                "kME":     float(kme_val),
            })

    return pd.DataFrame(rows).set_index("probe_id")


shoot_kme = compute_kme_table(shoot_wgcna, shoot_relevant_modules)
root_kme  = compute_kme_table(root_wgcna,  root_relevant_modules)

print(f"shoot kME table: {shoot_kme.shape[0]} genes across "
      f"{shoot_kme['module'].nunique()} modules")
print(f"root  kME table: {root_kme.shape[0]} genes across "
      f"{root_kme['module'].nunique()} modules")

print("shoot:", pd.read_parquet(OUTPUT_DIR / "shoot_hubs.parquet").index.name)
print("root: ", pd.read_parquet(OUTPUT_DIR / "root_hubs.parquet").index.name)

In [ ]:
# Quick look at the kME distribution within each stress-relevant module before
# the threshold gets applied in Section 4.

def kme_summary(kme_table):
    return (
        kme_table.groupby("module")["kME"]
        .agg(n="count", mean="mean", median="median", min="min", max="max")
        .sort_values("median", ascending=False)
        .round(3)
    )

print("=== shoot — kME distribution per stress-relevant module ===")
print(kme_summary(shoot_kme).to_string())
print()
print("=== root  — kME distribution per stress-relevant module ===")
print(kme_summary(root_kme).to_string())

The per-tissue kME tables are the input to Section 4, which applies the locked kME ≥ 0.8 threshold to select hub genes within each stress-relevant module.

Two observations worth carrying forward into the interpretation:

- **kME is signed.** Module eigengenes' signs are conventionally chosen so that the average member gene loads positively; for a gene in its assigned module, kME is typically positive. Negative or near-zero kME values indicate genes that are weakly or anti-aligned with the module they were assigned to.
- **Module-size effects are visible above.** Modules with many genes will generally show a wider kME spread than small modules; the threshold in Section 4 is fixed across modules and will surface this asymmetry as variation in hub-list size.

## 4) Apply the hub threshold

The kME distribution per module from Section 3 shows the raw material. This section applies the locked threshold to select hub genes within each stress-relevant module.

**Pre-commit (locked):** A gene is a **hub** in a stress-relevant module if its kME within that module is at least **0.8**. Applied per module per tissue.

The threshold of 0.8 follows WGCNA convention for hub identification — Langfelder & Horvath's framework and most downstream literature define hubs by an absolute kME threshold in the 0.7–0.8 range. The question page cites Kumar et al. (2023) as the methodological precedent for kME-based hub identification on *Arabidopsis* stress data. It is committed as a convention threshold, not as a value tuned to this dataset.

Two consequences worth flagging:

- **The threshold is on kME, not on |kME|.** Genes anti-aligned with their module's eigengene (negative kME) are not hubs by this definition. WGCNA's convention chooses each module eigengene's sign so the typical member gene loads positively; negative or near-zero kME indicates weak module membership.
- **Hub counts will vary widely across modules.** Section 3's distribution showed module maxima ranging from 0.543 to 0.957; modules with max below 0.8 will contribute zero hubs (`lightcoral` in shoot, `firebrick` in root). That's a finding — "stress-relevant by eigengene correlation but diffuse internally" — not a defect to fix by lowering the threshold.

In [ ]:
# Threshold hardcoded per the May 16 closeout lock.
# N0's precommit.json will be updated in a separate pass; until then, this
# value is not read from precommit["hub_identification"].
HUB_KME_THRESHOLD = 0.8


def select_hubs(kme_table, threshold=HUB_KME_THRESHOLD):
    """Filter kME table to hubs (kME >= threshold), add within-module rank."""
    hubs = kme_table[kme_table["kME"] >= threshold].copy()
    hubs["rank_in_module"] = (
        hubs.groupby("module")["kME"]
        .rank(method="min", ascending=False)
        .astype(int)
    )
    return hubs.sort_values(["module", "rank_in_module"])


shoot_hubs = select_hubs(shoot_kme)
root_hubs  = select_hubs(root_kme)

print(f"shoot hubs: {shoot_hubs.shape[0]} genes across "
      f"{shoot_hubs['module'].nunique()} modules")
print(f"root  hubs: {root_hubs.shape[0]} genes across "
      f"{root_hubs['module'].nunique()} modules")

In [ ]:
def hub_counts_by_module(hub_table, kme_table):
    """For each stress-relevant module, show n_hubs alongside the module's total size."""
    n_hubs = hub_table.groupby("module").size().rename("n_hubs")
    n_total = kme_table.groupby("module").size().rename("n_total")
    return (
        pd.concat([n_hubs, n_total], axis=1)
        .fillna(0)
        .astype({"n_hubs": int, "n_total": int})
        .assign(pct_hub=lambda d: (100 * d["n_hubs"] / d["n_total"]).round(1))
        .sort_values("n_hubs", ascending=False)
    )


print("=== shoot — hubs per stress-relevant module ===")
print(hub_counts_by_module(shoot_hubs, shoot_kme).to_string())
print()
print("=== root  — hubs per stress-relevant module ===")
print(hub_counts_by_module(root_hubs, root_kme).to_string())

In [ ]:
def show_top_hubs_per_module(hub_table, n_per_module=3):
    """Display the top n_per_module hubs from each module."""
    return (
        hub_table[hub_table["rank_in_module"] <= n_per_module]
        .sort_values(["module", "rank_in_module"])
        .round({"kME": 3})
    )


print("=== shoot — top 3 hubs per module ===")
print(show_top_hubs_per_module(shoot_hubs).to_string())
print()
print("=== root  — top 3 hubs per module ===")
print(show_top_hubs_per_module(root_hubs).to_string())

`shoot_hubs` and `root_hubs` are the gene-level result of this notebook's identification pipeline. Each row is a hub gene with its module assignment, kME score, and rank within its module. Section 5 joins these tables to the per-stress module-relevance information from Section 2 and saves the assembled tables to disk for Notebook 3.

## 5) Assemble and save per-tissue hub tables

The hub tables from Section 4 carry gene-level information (module, kME, rank). Two things still need to happen before they're ready for Notebook 3:

- **Join with per-stress associations.** Section 2 identified which stresses each module is relevant for; that information is added to each hub's row so downstream analysis can answer "which stresses is this gene a hub for?" without a separate join.
- **Save to disk.** `shoot_hubs.parquet` and `root_hubs.parquet` carry the per-gene information; `hub_summary.json` carries the counts and the thresholds used, for the Week 3 presentation.

In [ ]:
# For each tissue, build a module->stresses lookup from Section 2's candidate
# lists, then attach a comma-separated `stresses` column to each hub.

def module_to_stresses_map(candidates):
    """Invert {stress: [(ME-module, r), ...]} to {module: [stress, ...]} (no ME prefix)."""
    mapping = {}
    for stress, mods in candidates.items():
        for me_module, _ in mods:
            module = me_module[2:]  # strip "ME" prefix
            mapping.setdefault(module, []).append(stress)
    return {m: sorted(s) for m, s in mapping.items()}


shoot_module_stresses = module_to_stresses_map(shoot_candidates)
root_module_stresses  = module_to_stresses_map(root_candidates)

shoot_hubs["stresses"] = shoot_hubs["module"].map(
    lambda m: ", ".join(shoot_module_stresses[m])
)
root_hubs["stresses"] = root_hubs["module"].map(
    lambda m: ", ".join(root_module_stresses[m])
)

print("Hub table columns now:")
print(f"  shoot: {list(shoot_hubs.columns)}")
print(f"  root:  {list(root_hubs.columns)}")

print("\nExample rows from shoot_hubs:")
print(shoot_hubs.head(3).round({"kME": 3}).to_string())

In [ ]:
# Save as Parquet; round-trip read-back to confirm the saves are intact.

shoot_hubs_path = OUTPUT_DIR / "shoot_hubs.parquet"
root_hubs_path  = OUTPUT_DIR / "root_hubs.parquet"

shoot_hubs.to_parquet(shoot_hubs_path)
root_hubs.to_parquet(root_hubs_path)

assert pd.read_parquet(shoot_hubs_path).shape == shoot_hubs.shape
assert pd.read_parquet(root_hubs_path).shape == root_hubs.shape

print(f"Saved: {shoot_hubs_path}")
print(f"   ({shoot_hubs.shape[0]} hubs × {shoot_hubs.shape[1]} columns)")
print(f"Saved: {root_hubs_path}")
print(f"   ({root_hubs.shape[0]} hubs × {root_hubs.shape[1]} columns)")

In [ ]:
# Compact summary for the Week 3 presentation: hub counts and the thresholds used.

def hubs_per_stress(hub_table, module_stresses):
    """For each stress, count hubs whose module is relevant for that stress."""
    all_stresses = set().union(*module_stresses.values())
    counts = {}
    for stress in all_stresses:
        relevant_modules = {m for m, ss in module_stresses.items() if stress in ss}
        counts[stress] = int(hub_table[hub_table["module"].isin(relevant_modules)].shape[0])
    return dict(sorted(counts.items()))


hub_summary = {
    "thresholds": {
        "module_relevance_abs_pearson_r": MODULE_RELEVANCE_THRESHOLD,
        "hub_kme":                        HUB_KME_THRESHOLD,
    },
    "per_tissue": {
        "shoot": {
            "n_hubs":              int(shoot_hubs.shape[0]),
            "n_modules_with_hubs": int(shoot_hubs["module"].nunique()),
            "hubs_per_module":     {k: int(v) for k, v in shoot_hubs.groupby("module").size().items()},
            "hubs_per_stress":     hubs_per_stress(shoot_hubs, shoot_module_stresses),
        },
        "root": {
            "n_hubs":              int(root_hubs.shape[0]),
            "n_modules_with_hubs": int(root_hubs["module"].nunique()),
            "hubs_per_module":     {k: int(v) for k, v in root_hubs.groupby("module").size().items()},
            "hubs_per_stress":     hubs_per_stress(root_hubs, root_module_stresses),
        },
    },
}

hub_summary_path = OUTPUT_DIR / "hub_summary.json"
with open(hub_summary_path, "w") as f:
    json.dump(hub_summary, f, indent=2)

print(f"Saved: {hub_summary_path}")
print(f"  shoot: {hub_summary['per_tissue']['shoot']['n_hubs']} hubs across "
      f"{hub_summary['per_tissue']['shoot']['n_modules_with_hubs']} modules")
print(f"  root:  {hub_summary['per_tissue']['root']['n_hubs']} hubs across "
      f"{hub_summary['per_tissue']['root']['n_modules_with_hubs']} modules")

The per-tissue hub tables are saved and ready for Notebook 3's enrichment analysis. Section 6 closes this notebook with a cross-tissue overlap preview — which hub genes appear in both shoot and root, and what does that intersection look like by module?

## 6) Cross-tissue overlap preview

Each tissue produced its own per-tissue hub list. A natural follow-up question: are any probes hubs in *both* tissues? These would be tissue-independent stress-response signals — probes whose centrality to module-level co-expression survives across two independent network builds, and therefore the strongest individual candidates for the prediction the question page makes.

This section is a preview, not the full analysis. It surfaces three counts (shoot-only, root-only, both) and displays the top-ranked cross-tissue hubs by mean kME. Notebook 3's Section 6 picks up from here with the comparison against the known-regulators reference set (Hakkak 2026 Supp 3) — the genes that are hubs in both tissues *and* in the reference set are the strongest "we already knew them" findings; the genes that are hubs in both tissues *but not* in the reference set are the strongest data-driven candidates.

In [ ]:
# Cross-tissue overlap: probes that are hubs in both tissues.

shoot_hub_probes = set(shoot_hubs.index)
root_hub_probes  = set(root_hubs.index)

both       = shoot_hub_probes & root_hub_probes
shoot_only = shoot_hub_probes - root_hub_probes
root_only  = root_hub_probes  - shoot_hub_probes

print(f"Hubs in shoot only:        {len(shoot_only):>4}")
print(f"Hubs in root only:         {len(root_only):>4}")
print(f"Hubs in both tissues:      {len(both):>4}")
print(f"Total unique hub probes:   {len(shoot_hub_probes | root_hub_probes):>4}")

In [ ]:
# Side-by-side view of the probes that are hubs in both tissues.

both_sorted = sorted(both)
cross_tissue = (
    shoot_hubs.loc[both_sorted, ["module", "kME", "rank_in_module"]]
    .add_prefix("shoot_")
    .join(
        root_hubs.loc[both_sorted, ["module", "kME", "rank_in_module"]]
        .add_prefix("root_")
    )
)

# Sort by mean kME across tissues, highest first.
cross_tissue["mean_kME"] = (cross_tissue["shoot_kME"] + cross_tissue["root_kME"]) / 2
cross_tissue = cross_tissue.sort_values("mean_kME", ascending=False)

print(f"=== Top 15 cross-tissue hubs (by mean kME) ===")
print(
    cross_tissue.head(15)
    .round({"shoot_kME": 3, "root_kME": 3, "mean_kME": 3})
    .to_string()
)

The cross-tissue overlap is the headline result of this notebook's cross-tissue lens. Two things to take into Notebook 3:

- The both-tissues hub list above is the input to N3's reference-set comparison. A gene that is a hub in both shoot and root *and* in Hakkak 2026 Supp 3 is the strongest "known regulator" candidate; a gene that is a hub in both tissues *but not* in the reference set is the strongest data-driven candidate.
- The shoot-only and root-only hubs are not noise. A gene that is a hub in only one tissue may genuinely reflect tissue-specific stress biology — root-only hubs for water-uptake regulation, shoot-only hubs for UV-B response. These are real findings about tissue specialization, and Notebook 3's per-tissue enrichment tests treat them as such.

That's the end of Notebook 2. The per-tissue hub tables (`shoot_hubs.parquet`, `root_hubs.parquet`) and the summary (`hub_summary.json`) are saved to your R1-Q2 output directory; Notebook 3 picks them up from there.

"Virtual Lab-20260517 - R1-Q2 - N3 - Comparison"